# 📊 Step 6 of RAG — Benchmarking

This is the final notebook of Week 4. After building and refining our RAG pipeline, it is time to **measure** how well it actually performs.

We evaluate using **reference-based offline metrics** — no LLM-as-a-judge, no extra API calls. Since our test set contains ground-truth answers, we can score the system entirely locally:

```
alice_test_set.csv  (question + reference answer)
           │
           ▼
     RAG Pipeline
           │
           ▼
   { question, contexts, answer, reference }
           │
           ▼
      Offline Metrics
     ┌──────┴───────┐
  Retrieval      Generation
  • Hit Rate     • ROUGE-L
  • MRR          • BERTScore
```

| Metric | What it measures | LLM calls |
|---|---|---|
| **Hit Rate** | Does any retrieved chunk contain the reference answer? | 0 |
| **MRR** | How highly is the relevant chunk ranked? | 0 |
| **ROUGE-L** | Word-level overlap between answer and reference | 0 |
| **BERTScore** | Semantic similarity of answer vs reference | 0 (local model) |

## 0. Setup & Imports

In [ ]:
import os
import csv
import json
import random
import numpy as np
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from dotenv import load_dotenv
from munch import Munch

# Offline metrics
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Embedding model
from FlagEmbedding import BGEM3FlagModel

# LangChain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# Load config
config_path = (Path.cwd() / "../config/config.yaml").resolve()
with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

# Load secrets
load_dotenv((Path.cwd() / "../.env").resolve())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")

if not SUPABASE_PASSWORD:
    raise ValueError("SUPABASE_PASSWORD not found in .env")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

DATABASE_URL = (
    f"postgresql://{config.supabase.user}:{SUPABASE_PASSWORD}@"
    f"{config.supabase.host}:{config.supabase.port}/{config.supabase.database}"
)
sql_dir    = (Path.cwd() / "sql").resolve()
assets_dir = (Path.cwd() / "../assets").resolve()

print("✅ Config and secrets loaded")

### Load the embedding model

In [ ]:
print(f"Loading {config.embedding.model} …")
embed_model = BGEM3FlagModel(config.embedding.model, use_fp16=True)
print("✅ Embedding model loaded")

---

## 1. RAG Pipeline

We bring together the full pipeline from the previous notebooks: ensemble retrieval → LangChain RAG chain. Refer to Notebooks 4 and 5 for detailed explanations of each component.

In [ ]:
# ── Database helpers ───────────────────────────────────────────────────────────

def get_connection() -> psycopg2.extensions.connection:
    return psycopg2.connect(DATABASE_URL)


def retrieve_dense(q_dense: np.ndarray, top_k: int) -> list[dict]:
    sql = (sql_dir / "09_query_dense.sql").read_text()
    vec_str = f"[{','.join(map(str, q_dense.tolist()))}]"
    conn = get_connection()
    cur  = conn.cursor()
    try:
        cur.execute(sql, (vec_str, vec_str, top_k))
        return [{"chunk_id": r[0], "chunk_text": r[1], "score": float(r[2])} for r in cur.fetchall()]
    finally:
        cur.close(); conn.close()


def retrieve_sparse(q_sparse: dict, top_k: int) -> list[dict]:
    sql = (sql_dir / "10_query_sparse.sql").read_text()
    sparse_json = json.dumps({str(k): float(v) for k, v in q_sparse.items()})
    conn = get_connection()
    cur  = conn.cursor()
    try:
        cur.execute(sql, (sparse_json, top_k))
        return [{"chunk_id": r[0], "chunk_text": r[1], "score": float(r[2])} for r in cur.fetchall()]
    finally:
        cur.close(); conn.close()


def retrieve_colbert(q_colbert: np.ndarray, top_k: int) -> list[dict]:
    sql = (sql_dir / "11_query_colbert.sql").read_text()
    candidate_pool = top_k * 3
    chunk_scores: dict = {}
    conn = get_connection()
    cur  = conn.cursor()
    try:
        for token_vec in q_colbert:
            vec_str = f"[{','.join(map(str, token_vec.tolist()))}]"
            cur.execute(sql, (vec_str, candidate_pool))
            for chunk_id, chunk_text, max_sim in cur.fetchall():
                entry = chunk_scores.setdefault(chunk_id, {"chunk_text": chunk_text, "sims": []})
                entry["sims"].append(float(max_sim))
    finally:
        cur.close(); conn.close()
    results = [
        {"chunk_id": cid, "chunk_text": d["chunk_text"], "score": float(np.mean(d["sims"]))}
        for cid, d in chunk_scores.items()
    ]
    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


def ensemble_retrieve(query: str, top_k: int) -> list[dict]:
    """Embed query, run all three retrievers, fuse with RRF."""
    q_out = embed_model.encode(
        [query],
        max_length=config.embedding.max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True,
    )
    rrf_scores:  dict = {}
    chunk_texts: dict = {}
    for results in [
        retrieve_dense(q_dense=q_out["dense_vecs"][0], top_k=top_k),
        retrieve_sparse(q_sparse=q_out["lexical_weights"][0], top_k=top_k),
        retrieve_colbert(q_colbert=q_out["colbert_vecs"][0], top_k=top_k),
    ]:
        for rank, item in enumerate(results, start=1):
            cid = item["chunk_id"]
            rrf_scores[cid]  = rrf_scores.get(cid, 0.0) + 1.0 / (60 + rank)
            chunk_texts[cid] = item["chunk_text"]
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"chunk_id": cid, "chunk_text": chunk_texts[cid], "rrf_score": s} for cid, s in fused]


# ── LangChain RAG chain ────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using only the provided context.\n"
    "If the answer is not in the context, say \"I don't know based on the provided context.\"\n"
    "Be concise and accurate."
)

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=config.groq.model,
    temperature=config.groq.temperature,
    max_tokens=config.groq.max_tokens,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user",   "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {
        "context":  RunnableLambda(lambda x: x["context"]),
        "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"]),
    }
    | prompt
    | llm
    | StrOutputParser()
)


def format_context(chunks: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[Chunk {i + 1}]\n{c['chunk_text']}" for i, c in enumerate(chunks)
    )


def run_rag(question: str) -> dict:
    """
    Run the full RAG pipeline for a single question.

    Returns:
        Dict with keys: question, contexts (list[str]), answer.
    """
    chunks  = ensemble_retrieve(query=question, top_k=config.retrieval.top_k)
    context = format_context(chunks=chunks)
    answer  = rag_chain.invoke({"question": question, "context": context})
    return {
        "question": question,
        "contexts": [c["chunk_text"] for c in chunks],
        "answer":   answer,
    }


print("✅ RAG pipeline ready")

---

## 2. Load the Test Set

`alice_test_set.csv` contains hand-crafted question–answer pairs covering the full book. Each row has a `question` and a reference `answer` — exactly what we need for offline evaluation.

In [ ]:
def load_test_set(filepath: Path) -> list[dict]:
    """Load a CSV test set into a list of {question, answer} dicts."""
    with open(filepath, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))


full_test_set = load_test_set(
    filepath=assets_dir / "benchmark_datasets/alice_test_set.csv"
)
print(f"Total questions available: {len(full_test_set)}")
print(f"\nExample row:")
print(f"  Q: {full_test_set[0]['question']}")
print(f"  A: {full_test_set[0]['answer']}")

In [ ]:
# Sample a subset to keep Groq usage reasonable
# One Groq call per question — 20 questions = 20 calls total
SAMPLE_SIZE = config.benchmarking.sample_size

random.seed(config.benchmarking.random_seed)
test_sample = random.sample(full_test_set, k=min(SAMPLE_SIZE, len(full_test_set)))

print(f"Sampled {len(test_sample)} questions for evaluation")

---

## 3. Collect RAG Outputs

We run the RAG pipeline on each sampled question. This is the only step that uses the Groq API — one call per question.

In [ ]:
def collect_rag_outputs(test_samples: list[dict]) -> list[dict]:
    """
    Run the RAG pipeline on every question and attach the reference answer.

    Args:
        test_samples : List of {question, answer, ...} dicts from the test CSV.

    Returns:
        List of {question, contexts, answer, reference} dicts.
    """
    outputs = []
    for i, row in enumerate(test_samples, start=1):
        print(f"  [{i}/{len(test_samples)}] {row['question'][:70]}…")
        result = run_rag(question=row["question"])
        outputs.append({
            "question":  row["question"],
            "contexts":  result["contexts"],
            "answer":    result["answer"],
            "reference": row["answer"],
        })
    return outputs


print("Running RAG pipeline …\n")
outputs = collect_rag_outputs(test_samples=test_sample)
print("\n✅ RAG outputs collected")

---

## 4. Offline Metrics

### 4.1 Retrieval metrics

**Context Hit Rate** and **MRR** measure whether the retriever surfaces the right content. Rather than exact string matching (which fails because the reference is a clean summary while chunks contain raw prose), we use **ROUGE-L overlap** between the reference and each chunk as a soft relevance signal.

A chunk is considered a hit if its ROUGE-L score against the reference exceeds a threshold.

In [ ]:
RETRIEVAL_THRESHOLD = config.benchmarking.retrieval_threshold


def chunk_rouge_l(chunk: str, reference: str) -> float:
    """ROUGE-L F1 between a single chunk and the reference answer."""
    scorer = rouge_scorer.RougeScorer(rouge_types=["rougeL"], use_stemmer=True)
    return scorer.score(target=reference, prediction=chunk)["rougeL"].fmeasure


def hit_rate(contexts: list[str], reference: str) -> float:
    """
    1.0 if any chunk's ROUGE-L against the reference exceeds the threshold.

    Args:
        contexts  : Retrieved chunk texts.
        reference : Ground-truth answer string.
    """
    return float(
        any(chunk_rouge_l(chunk=ctx, reference=reference) >= RETRIEVAL_THRESHOLD
            for ctx in contexts)
    )


def mean_reciprocal_rank(contexts: list[str], reference: str) -> float:
    """
    Reciprocal rank of the first chunk whose ROUGE-L exceeds the threshold.

    Args:
        contexts  : Retrieved chunk texts, ordered by rank.
        reference : Ground-truth answer string.
    """
    for rank, ctx in enumerate(contexts, start=1):
        if chunk_rouge_l(chunk=ctx, reference=reference) >= RETRIEVAL_THRESHOLD:
            return 1.0 / rank
    return 0.0


print(f"✅ Retrieval metric functions defined  (threshold = {RETRIEVAL_THRESHOLD})")

### 4.2 Generation metrics

**ROUGE-L** measures the longest common subsequence between the answer and the reference — it captures word-level overlap without caring about word order.

**BERTScore** uses a pre-trained language model to compare answer and reference at the *semantic* level. Two answers that mean the same thing but use different words will score high on BERTScore but low on ROUGE-L — making the two metrics complementary.

In [ ]:
def rouge_l(prediction: str, reference: str) -> float:
    """
    ROUGE-L F1 score between a predicted answer and the reference.

    Args:
        prediction : Generated answer string.
        reference  : Ground-truth answer string.
    """
    scorer = rouge_scorer.RougeScorer(rouge_types=["rougeL"], use_stemmer=True)
    return scorer.score(target=reference, prediction=prediction)["rougeL"].fmeasure


def compute_bert_scores(predictions: list[str], references: list[str]) -> list[float]:
    """
    Batch BERTScore F1 — semantic similarity between predictions and references.

    Args:
        predictions : List of generated answer strings.
        references  : List of ground-truth answer strings.

    Returns:
        List of per-sample F1 scores.
    """
    _, _, f1 = bert_score(
        cands=predictions,
        refs=references,
        lang="en",
        verbose=False,
    )
    return f1.tolist()


print("✅ Generation metric functions defined")

---

## 5. Run Evaluation

We compute all four metrics across every sample. BERTScore is batched for efficiency — it loads the model once and scores all samples in one pass.

In [ ]:
def evaluate_outputs(outputs: list[dict]) -> pd.DataFrame:
    """
    Run all offline metrics on a list of RAG outputs.

    Args:
        outputs : List of {question, contexts, answer, reference} dicts.

    Returns:
        DataFrame with one row per sample and one column per metric.
    """
    total_samples = len(outputs)
    print(f"📊 Starting evaluation on {total_samples} samples...")
    print("=" * 70)
    
    predictions = [o["answer"]    for o in outputs]
    references  = [o["reference"] for o in outputs]

    print("\n🔍 Computing BERTScore (runs locally, may take a while)...")
    bert_scores = compute_bert_scores(predictions=predictions, references=references)
    print("✅ BERTScore computation complete")

    print(f"\n📈 Computing per-sample metrics...")
    rows = []
    
    # Progress milestones (every 10%)
    milestones = set(range(0, total_samples, max(1, total_samples // 10)))
    
    for i, o in enumerate(outputs):
        # Print progress at milestones
        if i in milestones or i == total_samples - 1:
            progress = (i + 1) / total_samples * 100
            print(f"  Progress: {i + 1}/{total_samples} ({progress:.0f}%)")
        
        rows.append({
            "question":    o["question"],
            "reference":   o["reference"],
            "answer":      o["answer"],
            "hit_rate":    hit_rate(contexts=o["contexts"], reference=o["reference"]),
            "mrr":         mean_reciprocal_rank(contexts=o["contexts"], reference=o["reference"]),
            "rouge_l":     rouge_l(prediction=o["answer"], reference=o["reference"]),
            "bert_score":  bert_scores[i],
        })

    print("\n" + "=" * 70)
    print("✅ Evaluation complete!")
    print(f"📋 Processed {total_samples} samples with 4 metrics each")
    
    return pd.DataFrame(rows)


results_df = evaluate_outputs(outputs=outputs)


---

## 6. Results

### 6.1 Score Summary

In [ ]:
METRIC_COLS = ["hit_rate", "mrr", "rouge_l", "bert_score"]

summary = results_df[METRIC_COLS].mean().rename("mean").to_frame().T.round(3)
print("Mean scores across all samples:\n")
print(summary.to_string(index=False))

### 6.2 Visualisation

In [ ]:
matplotlib.rcParams["font.family"] = "monospace"

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 4))
fig.suptitle("RAG Benchmark — Alice's Adventures in Wonderland", fontsize=13)

palette = ["#5b8dd9", "#6dd9a0", "#c9a96e", "#d9a85b"]

# Left: mean bar chart
ax = axes[0]
means = results_df[METRIC_COLS].mean()
bars  = ax.bar(METRIC_COLS, means, color=palette, width=0.5, zorder=3)
ax.set_ylim(0, 1.12)
ax.set_title("Mean scores", fontsize=11)
ax.set_ylabel("Score (0–1)")
ax.set_xticklabels([m.replace("_", "\n") for m in METRIC_COLS], fontsize=9)
ax.yaxis.grid(visible=True, linestyle="--", alpha=0.4, zorder=0)
ax.set_axisbelow(True)
for bar, val in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.2f}",
        ha="center", va="bottom", fontsize=9,
    )

# Right: per-sample score distribution (box plot)
ax2 = axes[1]
ax2.boxplot(
    [results_df[col].tolist() for col in METRIC_COLS],
    labels=[m.replace("_", "\n") for m in METRIC_COLS],
    patch_artist=True,
    boxprops=dict(facecolor="#1c1c1c", color="#5b8dd9"),
    medianprops=dict(color="#c9a96e", linewidth=2),
    whiskerprops=dict(color="#6b6560"),
    capprops=dict(color="#6b6560"),
    flierprops=dict(marker="o", color="#6b6560", markersize=4),
)
ax2.set_ylim(0, 1.12)
ax2.set_title("Score distribution", fontsize=11)
ax2.set_ylabel("Score (0–1)")
ax2.yaxis.grid(visible=True, linestyle="--", alpha=0.4, zorder=0)
ax2.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 6.3 Per-sample Breakdown

Inspecting the lowest-scoring samples tells us *why* the system fails — a low ROUGE-L alongside a high BERTScore suggests the answer is correct but phrased differently; a low Hit Rate points to a retrieval failure.

In [ ]:
# 3 samples where the retriever failed to find the answer
print("── Retrieval failures (hit_rate = 0) ──")
misses = results_df[results_df["hit_rate"] == 0][["question", "reference", "answer", "rouge_l", "bert_score"]]
print(f"  {len(misses)} / {len(results_df)} questions had no context hit\n")
for _, row in misses.head(3).iterrows():
    print(f"  Q  : {row['question']}")
    print(f"  Ref: {row['reference']}")
    print(f"  Ans: {row['answer']}")
    print(f"  ROUGE-L: {row['rouge_l']:.3f}  |  BERTScore: {row['bert_score']:.3f}\n")

In [ ]:
# 3 lowest ROUGE-L samples — where the answer wording diverges most from the reference
print("── Lowest ROUGE-L samples ──")
worst_rouge = results_df.nsmallest(3, "rouge_l")[["question", "reference", "answer", "rouge_l", "bert_score"]]
for _, row in worst_rouge.iterrows():
    print(f"  Q  : {row['question'][:80]}")
    print(f"  Ref: {row['reference']}")
    print(f"  Ans: {row['answer'][:100]}")
    print(f"  ROUGE-L: {row['rouge_l']:.3f}  |  BERTScore: {row['bert_score']:.3f}\n")

---

## 7. Save Results

In [ ]:
output_dir = (Path.cwd() / "../assets/benchmark_results").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "alice_benchmark_results.csv", index=False)

print(f"✅ Results saved to {output_dir / 'alice_benchmark_results.csv'}")

---

## 8. Summary & Week 4 Wrap-up

### Metric reference

| Metric | Dimension | Interpretation |
|---|---|---|
| **Hit Rate** | Retrieval | Fraction of questions where the right content was retrieved |
| **MRR** | Retrieval | How early in the ranking the right content appears |
| **ROUGE-L** | Generation | Word-level overlap with the reference answer |
| **BERTScore** | Generation | Semantic similarity with the reference answer |

### How to read the scores

- **Low Hit Rate** → the retriever is missing relevant chunks — try larger `top_k`, better chunking, or re-ranking (Notebook 6).
- **Hit Rate high, ROUGE-L low** → the right content was retrieved but the LLM paraphrased it heavily — check the system prompt.
- **ROUGE-L low, BERTScore high** → the answer is semantically correct but worded differently from the reference — this is often fine.
- **Both low** → retrieval failure or hallucination — inspect the per-sample breakdown above.

### The complete Week 4 pipeline

| Notebook | What it covered |
|---|---|
| `2 - Read Chunk and Embed` | Read, clean, chunk, embed with BGE-M3 |
| `3 - Store Embeddings` | Persist dense, sparse, ColBERT in PostgreSQL + pgvector |
| `4 - Vector DB Retrieval` | Query and compare all three retrieval methods |
| `5 - RAG` | LangChain LCEL chain: ensemble retrieval → Groq/LLaMA → answer |
| `6 - Re-ranking` | Cross-encoder and MMR re-ranking over retrieval candidates |
| `7 - Benchmarking` | Offline evaluation with Hit Rate, MRR, ROUGE-L, BERTScore |

> 💡 **Next step**: use this notebook as a regression test. Every time you change chunking strategy, `top_k`, or re-ranking, re-run it and compare the CSV outputs to track whether the change helped.